In [9]:
# Colab-ready: two animations (total field, and separate waves), widescreen, optional GIFs
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, Image
from matplotlib.animation import PillowWriter

# -----------------------------
# Parameters (edit as needed)
# -----------------------------
# Medium 1 (air)
c1 = 343.0
rho1 = 1.2
Z1 = rho1 * c1

# Medium 2 (wood, approximate)
c2 = 3300.0
rho2 = 600.0
Z2 = rho2 * c2

# Frequency and grids
f = 1000.0
omega = 2.0 * np.pi * f
x = np.linspace(-2.0, 2.0, 600)      # more points for smoother curves
t = np.linspace(0.0, 0.02, 240)      # more frames for smoother animation

# Wide figure size (width, height) in inches
FIGSIZE = (16, 4)

# Save GIFs?
SAVE_GIF = True
GIF_FPS = 20

# -----------------------------
# Coefficients (pressure amplitude)
# -----------------------------
Rp = (Z2 - Z1) / (Z2 + Z1)
Tp = 2.0 * Z2 / (Z2 + Z1)

# -----------------------------
# Fields
# -----------------------------
X, T = np.meshgrid(x, t)
k1 = omega / c1
k2 = omega / c2

# Incident and reflected (left side, x < 0)
incident = np.sin(omega * T - k1 * X)
reflected = Rp * np.sin(omega * T + k1 * X)

# Transmitted (right side, x > 0)
transmitted = Tp * np.sin(omega * T - k2 * X)

# Total field (piecewise)
field_total = np.where(X < 0.0, incident + reflected, transmitted)

# For the "separate waves" animation, mask each wave outside its physical region
mask_left  = (X < 0.0).astype(float)
mask_right = (X > 0.0).astype(float)

incident_masked   = np.where(mask_left  > 0.5, incident,   np.nan)
reflected_masked  = np.where(mask_left  > 0.5, reflected,  np.nan)
transmit_masked   = np.where(mask_right > 0.5, transmitted,np.nan)

# Common axis limits
ymin, ymax = -2.0, 2.0

# ============================================================
# Animation 1: Total field (incident+reflected on left; transmitted on right)
# ============================================================
fig1, ax1 = plt.subplots(figsize=FIGSIZE)
line_total, = ax1.plot([], [], lw=2)
ax1.axvline(0.0, color='k', linestyle='--', label='Interface')
ax1.set_xlim(x.min(), x.max())
ax1.set_ylim(ymin, ymax)
ax1.set_xlabel('x (m)')
ax1.set_ylabel('Amplitude (a.u.)')
ax1.set_title('Campo total: incidente+refletida (x<0) e transmitida (x>0)')
ax1.legend(loc='upper right')

def init_total():
    # Initialize empty line
    line_total.set_data([], [])
    return (line_total,)

def animate_total(i):
    # Update total field at frame i
    line_total.set_data(x, field_total[i, :])
    return (line_total,)

ani_total = animation.FuncAnimation(
    fig1, animate_total, init_func=init_total,
    frames=len(t), interval=50, blit=True
)
plt.close(fig1)  # avoid duplicate static fig in Colab output

display(HTML(ani_total.to_jshtml()))

if SAVE_GIF:
    gif1 = 'onda_total.gif'
    ani_total.save(gif1, writer=PillowWriter(fps=GIF_FPS))
    display(Image(gif1))

# ============================================================
# Animation 2: Separate waves (incident, reflected, transmitted) — no sum
# ============================================================
fig2, ax2 = plt.subplots(figsize=FIGSIZE)
line_inc,  = ax2.plot([], [], lw=2, label='Incidente')
line_refl, = ax2.plot([], [], lw=2, label='Refletida')
line_tran, = ax2.plot([], [], lw=2, label='Transmitida')
ax2.axvline(0.0, color='k', linestyle='--', label='Interface')
ax2.set_xlim(x.min(), x.max())
ax2.set_ylim(ymin, ymax)
ax2.set_xlabel('x (m)')
ax2.set_ylabel('Amplitude (a.u.)')
ax2.set_title('Ondas separadas: incidente (x<0), refletida (x<0), transmitida (x>0)')
ax2.legend(loc='upper right')

def init_sep():
    # Initialize empty lines
    line_inc.set_data([], [])
    line_refl.set_data([], [])
    line_tran.set_data([], [])
    return (line_inc, line_refl, line_tran)

def animate_sep(i):
    # Update each wave independently (masked outside its region)
    line_inc.set_data(x,  incident_masked[i, :])
    line_refl.set_data(x, reflected_masked[i, :])
    line_tran.set_data(x,    transmit_masked[i, :])
    return (line_inc, line_refl, line_tran)

ani_sep = animation.FuncAnimation(
    fig2, animate_sep, init_func=init_sep,
    frames=len(t), interval=50, blit=True
)
plt.close(fig2)

display(HTML(ani_sep.to_jshtml()))

if SAVE_GIF:
    gif2 = 'ondas_separadas.gif'
    ani_sep.save(gif2, writer=PillowWriter(fps=GIF_FPS))
    display(Image(gif2))


Output hidden; open in https://colab.research.google.com to view.